Используемый датасет для задания "Распознавание погоды по видео с камер наблюдения": Weather Image Recognition (https://www.kaggle.com/datasets/jehanbhathena/weather-datase)

In [ ]:
# =========================
# WEATHER VIDEO (IMAGE) LABELING PIPELINE
# Kaggle: Weather Image Recognition
# =========================

import os
import zipfile
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.models import resnet18

from sklearn.model_selection import train_test_split

# -------------------------
# 1. DOWNLOAD DATASET FROM KAGGLE
# -------------------------
!pip install -q kaggle

# Expect kaggle.json in /content or ~/.kaggle/
os.makedirs("/root/.kaggle", exist_ok=True)

DATASET = "jehanbhathena/weather-dataset"
DATA_DIR = "weather_data"

!kaggle datasets download -d {DATASET} -p {DATA_DIR} --unzip

# -------------------------
# 2. LOAD DATASET
# -------------------------
root_dir = Path(DATA_DIR)

# find image root folder automatically
img_root = None
for p in root_dir.rglob("*"):
    if p.is_dir():
        # heuristic: folder contains subfolders = classes
        subdirs = [x for x in p.iterdir() if x.is_dir()]
        if len(subdirs) >= 5:
            img_root = p
            break

print("Detected image root:", img_root)

dataset = ImageFolder(img_root)

class_names = dataset.classes
num_classes = len(class_names)

print("Classes:", class_names)

# -------------------------
# 3. SPLIT DATA (LABELED / UNLABELED SIMULATION)
# -------------------------
indices = list(range(len(dataset)))
train_idx, test_idx = train_test_split(indices, test_size=0.3, random_state=42)

# simulate manual labeling subset (5%)
labeled_idx = random.sample(train_idx, int(0.05 * len(train_idx)))
unlabeled_idx = list(set(train_idx) - set(labeled_idx))

print("Labeled samples:", len(labeled_idx))
print("Unlabeled samples:", len(unlabeled_idx))

# -------------------------
# 4. DATASET WRAPPER
# -------------------------
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

class CustomDS(Dataset):
    def __init__(self, base_ds, indices):
        self.base = base_ds
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        img, label = self.base[self.indices[i]]
        return transform(img), label

labeled_ds = CustomDS(dataset, labeled_idx)
unlabeled_ds = CustomDS(dataset, unlabeled_idx)
test_ds = CustomDS(dataset, test_idx)

train_loader = DataLoader(labeled_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32)

# -------------------------
# 5. MODEL (TRANSFER LEARNING)
# -------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"

model = resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# -------------------------
# 6. TRAIN ON SMALL LABELED SET
# -------------------------
def train_one_epoch():
    model.train()
    total_loss = 0
    for x, y in tqdm(train_loader):
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        pred = model(x)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    return total_loss / len(train_loader)

for epoch in range(3):  # quick demo training
    loss = train_one_epoch()
    print(f"Epoch {epoch}: loss={loss:.4f}")

# -------------------------
# 7. PSEUDO-LABEL UNLABELED DATA
# -------------------------
model.eval()

pseudo_labels = []
pseudo_indices = []

with torch.no_grad():
    for i in tqdm(unlabeled_idx[:2000]):  # limit for speed
        img, _ = dataset[i]
        img = transform(img).unsqueeze(0).to(device)

        pred = model(img)
        label = torch.argmax(pred, dim=1).item()

        pseudo_labels.append(label)
        pseudo_indices.append(i)

print("Pseudo-labeled samples:", len(pseudo_labels))

# -------------------------
# 8. SAVE PSEUDO LABELED DATA
# -------------------------
pseudo_df = pd.DataFrame({
    "index": pseudo_indices,
    "pseudo_label": pseudo_labels,
    "class_name": [class_names[i] for i in pseudo_labels]
})

pseudo_df.to_csv("pseudo_labels.csv", index=False)

# -------------------------
# 9. CREATE HUMAN LABELING GUIDE
# -------------------------
guide = f"""
WEATHER VIDEO/IMAGE LABELING GUIDE

Dataset: Weather Image Recognition
Classes ({num_classes}):
{class_names}

TASK:
Label each frame/image with ONE class.

LABEL DEFINITIONS:

- dew: moisture on surfaces, morning condensation
- fog/smog: low visibility haze
- frost: ice crystals on surfaces
- glaze: smooth ice coating
- hail: ice pellets falling
- lightning: visible lightning strikes
- rain: rainfall visible
- rainbow: visible rainbow arcs
- rime: frozen fog deposits
- sandstorm: dust/sand in air
- snow: falling snow

RULES:
1. Choose only one label per image
2. If multiple phenomena appear → choose dominant weather
3. If unclear → mark as "fog/smog"
4. Ignore camera artifacts (noise, blur)

QUALITY CONTROL:
- 10% of images should be double-checked
- disagreements resolved by senior annotator

OUTPUT FORMAT:
image_id, label
"""

with open("labeling_guide.txt", "w") as f:
    f.write(guide)

print("DONE: pseudo_labels.csv + labeling_guide.txt created")

# -------------------------
# 10. SIMPLE EVALUATION
# -------------------------
correct = 0
total = 0

with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        pred = model(x)
        pred = torch.argmax(pred, dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)

print("Test accuracy:", correct / total)